# 02 · Experiments (Exp 0–6, baselines)

Thin driver: logic lives in `src/adgraph`, every setting is in `configs/experiments/`. See `docs/experiments.md` and `DESIGN.md`.

Run order: tests → dry run (inputs, shapes, splits; no training) → the settings you choose. Outputs go to a new timestamped folder under `processed/experiments/`; nothing is overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BRANCH = 'claude/ecstatic-davinci-a2e8mh'  # change to main once merged
!rm -rf /content/repo && git clone -q -b $BRANCH https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
!git rev-parse HEAD

## 1. Synthetic test suite (no EEG)

In [ ]:
!python -m pytest -q

## 2. Choose the graph run
The most recently assembled full graph run (smoke tests and unfinished runs are excluded).

In [ ]:
import glob
from adgraph.pipeline import latest_complete_graph_run, list_graph_runs
print(list_graph_runs('/content/drive/MyDrive/AD-LEB/processed').to_string())
GRAPHS = str(latest_complete_graph_run('/content/drive/MyDrive/AD-LEB/processed'))  # or set explicitly
OUT = '/content/drive/MyDrive/AD-LEB/processed/experiments'
print(GRAPHS)

## 3. Dry run: shapes, subject/window counts and split checks for every variant (no training)

In [ ]:
for exp in ['exp0_sensor_psd', 'exp1_source_psd', 'exp2_conn_minimal', 'exp3_conn_psd', 'exp3b_conn_psd_allbands', 'exp4_band_mean',
            'exp5_multiband', 'exp6_multiband_psd_strength', 'baseline_age', 'baseline_conn_classical',
            'age/exp1_source_psd_plus_age', 'age/exp6_multiband_psd_strength_plus_age']:
    !python scripts/run_experiment.py --config configs/experiments/{exp}.yaml --graphs {GRAPHS} --dry-run --settings within_D1 within_D2 pooled

## 4. Runs (resumable; uncomment deliberately)
Rerunning the same command after a disconnect continues unfinished folds/repeats; completed ones are never retrained. `--status` shows progress without running. Cross-dataset settings are refused automatically unless the graph run passes the DESIGN.md §7 gate.

In [ ]:
# !python scripts/run_experiment.py --config configs/experiments/exp0_sensor_psd.yaml --graphs {GRAPHS} --out {OUT} --settings within_D1 within_D2 --status
# !python scripts/run_experiment.py --config configs/experiments/exp0_sensor_psd.yaml --graphs {GRAPHS} --out {OUT} --settings within_D1 within_D2
# !python scripts/run_experiment.py --config configs/experiments/exp5_multiband.yaml --graphs {GRAPHS} --out {OUT} --settings within_D1 within_D2 --variants plv